# 03a. Разметка: экзамен и 400 фотографий

Последовательность работы: подключение → создание проекта и навыков → создание пулов → загрузка заданий. HTML, CSS, JavaScript и инструкции находятся в локальных файлах.

Запускайте из корня репозитория. Зависимости: `python -m pip install -r requirements-tasks.txt`. OAuth-токен заказчика задайте в локальном `.env` как `YANDEX_TASKS_OAUTH_TOKEN`.

Ноутбук создаёт проект, навыки и пулы, затем загружает задания. Открытие пулов, проверка ответов и назначение навыков исполнителям выполняются отдельно.

In [ ]:
import json
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path
from uuid import uuid4

import requests
from dotenv import load_dotenv

load_dotenv()
api = requests.Session()
api.headers['Authorization'] = f"OAuth {os.environ['YANDEX_TASKS_OAUTH_TOKEN']}"
BASE_URL = 'https://tasks.yandex.ru/api/v1'


def create(path, payload, params=None):
    response = api.post(f'{BASE_URL}/{path}', json=payload, params=params, timeout=60)
    response.raise_for_status()
    return response.json()


## Проект

Интерфейс разметки и инструкция находятся в `interface/annotation/`.

In [ ]:
interface = Path('interface/annotation')

view_spec = {'type': 'classic',
 'assets': {'style_urls': [],
            'script_urls': ['$TOLOKA_ASSETS/js/toloka-handlebars-templates.js',
                            '$TOLOKA_ASSETS/js/image-annotation.js']},
 'markup': (interface / 'task.html').read_text(encoding='utf-8'),
 'script': (interface / 'task.js').read_text(encoding='utf-8'),
 'styles': (interface / 'task.css').read_text(encoding='utf-8'),
 'settings': {'showSkip': True,
              'showTimer': True,
              'showTitle': True,
              'showFinish': True,
              'showReward': True,
              'showSubmit': True,
              'permissions': [],
              'showMessage': True,
              'showFullscreen': True,
              'showInstructions': True}}

project_payload = {'public_name': 'Разметка бабочек на фотографии',
 'public_description': 'Нарисуйте прямоугольные рамки вокруг взрослых бабочек на фотографии',
 'public_instructions': (interface / 'instructions.html').read_text(encoding='utf-8'),
 'private_comment': 'Финальный проект по разметке шад 2026',
 'assignments_issuing_type': 'AUTOMATED',
 'assignments_automerge_enabled': False,
 'task_spec': {'input_spec': {'image': {'type': 'url', 'hidden': False, 'required': True},
                              'photo_id': {'type': 'integer', 'hidden': True, 'required': True}},
               'output_spec': {'status': {'type': 'string',
                                          'hidden': False,
                                          'required': True,
                                          'allowed_values': ['FOUND',
                                                             'NONE',
                                                             'LOAD_ERROR',
                                                             'UNCERTAIN']},
                               'annotations': {'type': 'json', 'hidden': False, 'required': False}},
               'view_spec': view_spec}}

In [ ]:
project = create('projects', project_payload)
project_id = project['id']
project_id

## Навыки

Результат экзамена и участие в нём — разные навыки. В рабочую разметку допускаются с результатом от 80. Навык участия исключает разметчиков из валидации.

In [ ]:
exam_skill = create('skills', {'name': f'BBox — экзамен ({project_id})',
 'private_comment': 'Доля правильно размеченных фото на экзамене bbox после проверки командой.',
 'hidden': True,
 'public_name': {'EN': 'BBox — экзамен (11688)'}})
exam_skill_id = exam_skill['id']

participation_skill = create('skills', {'name': f'BBox — участвовал в экзамене ({project_id})',
 'private_comment': 'Получал страницу bbox-экзамена, независимо от результата; исключить из обоих '
                    'пулов ревью.',
 'hidden': True,
 'public_name': {'EN': 'BBox — участвовал в экзамене (11688)'}})
participation_skill_id = participation_skill['id']

## Экзамен разметчиков

Параметры: 2 ₽ за страницу, перекрытие 40, лимит 20 минут.

In [ ]:
exam_pool_payload = {'project_id': project_id,
 'private_name': 'BBox — экзамен',
 'private_comment': 'Пилот экзамена bbox, одна страница, оценка по локальному ключу.',
 'public_description': 'Экзамен по разметке бабочек: 10 фото. Допуск: 8 верных ответов.',
 'may_contain_adult_content': False,
 'reward_per_assignment': 2.0,
 'assignment_max_duration_seconds': 1200,
 'auto_accept_solutions': True,
 'auto_accept_period_day': 21,
 'assignments_issuing_config': {'issue_task_suites_in_creation_order': False},
 'filter': {'and': [{'or': [{'category': 'computed',
                             'key': 'client_type',
                             'operator': 'EQ',
                             'value': 'BROWSER'}]},
                    {'or': [{'category': 'profile',
                             'key': 'languages',
                             'operator': 'IN',
                             'value': 'RU'}]}]},
 'quality_control': {'configs': [{'collector_config': {'type': 'ANSWER_COUNT', 'parameters': {}},
                                  'rules': [{'conditions': [{'key': 'assignments_accepted_count',
                                                             'operator': 'GTE',
                                                             'value': 1}],
                                             'action': {'type': 'RESTRICTION_V2',
                                                        'parameters': {'scope': 'POOL',
                                                                       'duration_unit': 'PERMANENT'}}}]}]},
 'defaults': {'default_overlap_for_new_task_suites': 40},
 'speed_quality_balance': {'percent': 100, 'type': 'TOP_PERCENTAGE_BY_QUALITY'},
 'will_expire': (datetime.now(timezone.utc) + timedelta(days=14)).strftime('%Y-%m-%dT%H:%M:%SZ')}

In [ ]:
exam_pool = create('pools', exam_pool_payload)
exam_pool_id = exam_pool['id']
exam_pool_id

### Загрузка заданий

10 экзаменационных фотографий на одной странице, перекрытие 40. Эталонные рамки остаются локально в `tasks/annotation_exam_key.jsonl`: редактор рамок проверяется отдельно, в задания ключ не передаётся.

In [ ]:
pages = json.loads(Path('tasks/annotation_exam.json').read_text(encoding='utf-8'))
for page in pages:
    create('task-suites', {'pool_id': exam_pool_id, **page}, params={'open_pool': 'false'})

print(f'Загружено страниц: {len(pages)}; заданий: {sum(len(p["tasks"]) for p in pages)}')

Экзамен оформлен обычным пулом, поскольку рамки проверяются отдельно по эталонам. `ANSWER_COUNT` ограничивает исполнителя одной принятой страницей. Результат экзамена и навык участия назначаются отдельно; эти операции в ноутбук не входят.

## Рабочая разметка 400 фотографий

Параметры: 2 ₽ за страницу, перекрытие 1, лимит 15 минут.

In [ ]:
work_pool_payload = {'project_id': project_id,
 'private_name': 'BBox — рабочая разметка 400 фото',
 'private_comment': '400 основных фото; 100 страниц по 4; отдельно ревью; приемка 14 дней.',
 'public_description': 'Разметка взрослых бабочек: 4 фотографии на странице.',
 'may_contain_adult_content': False,
 'reward_per_assignment': 2.0,
 'assignment_max_duration_seconds': 900,
 'auto_accept_solutions': False,
 'auto_accept_period_day': 14,
 'assignments_issuing_config': {'issue_task_suites_in_creation_order': False},
 'filter': {'and': [{'or': [{'category': 'computed',
                             'key': 'client_type',
                             'operator': 'EQ',
                             'value': 'BROWSER'}]},
                    {'or': [{'category': 'profile',
                             'key': 'languages',
                             'operator': 'IN',
                             'value': 'RU'}]},
                    {'category': 'skill',
                     'key': exam_skill_id,
                     'operator': 'GTE',
                     'value': '80'}]},
 'defaults': {'default_overlap_for_new_task_suites': 1},
 'will_expire': (datetime.now(timezone.utc) + timedelta(days=30)).strftime('%Y-%m-%dT%H:%M:%SZ')}

In [ ]:
work_pool = create('pools', work_pool_payload)
work_pool_id = work_pool['id']
work_pool_id

### Загрузка заданий

400 фотографий, 100 страниц по 4 задания. Эталонные ответы не передаются.

In [ ]:
pages = json.loads(Path('tasks/annotation_work.json').read_text(encoding='utf-8'))
for page in pages:
    create('task-suites', {'pool_id': work_pool_id, **page}, params={'open_pool': 'false'})

print(f'Загружено страниц: {len(pages)}; заданий: {sum(len(p["tasks"]) for p in pages)}')

## Созданные ID

Сохраняем проект, навыки и пулы в `annotation_ids.json`. Эти ID используют ноутбуки валидации и переразметки.

In [ ]:
annotation_ids = {
    'project_id': project_id,
    'exam_skill_id': exam_skill_id,
    'participation_skill_id': participation_skill_id,
    'exam_pool_id': exam_pool_id,
    'work_pool_id': work_pool_id,
}
Path('annotation_ids.json').write_text(
    json.dumps(annotation_ids, ensure_ascii=False, indent=2), encoding='utf-8'
)
annotation_ids